# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Kingsley Wunzooya Nahyi
**Student ID:** 84552028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [2]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv

from groq import Groq

load_dotenv(override=True)  # Load .env file if it exists, and override any existing env vars.
print(os.getcwd())

API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---

API_KEY = os.getenv("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):

client = Groq(api_key=API_KEY)
MODEL = "llama-3.3-70b-versatile"




e:\COMPUTING\programming\New folder\%USERPROFILE%\ai_labs\lab-4


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [3]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#


def ask_llm(user_prompt, system_prompt="You are a helpful assistant.", temperature=0.7, max_tokens=500):
   response = client.chat.completions.create(
      model=MODEL,
      messages=[
         {"role": "system", "content": system_prompt},
         {"role": "user", "content": user_prompt}
      ],
      temperature=temperature,
      max_tokens=max_tokens
   )
   return response

response = ask_llm("What is the capital of France?")
print(response.choices[0].message.content)
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

The capital of France is Paris.


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

The system prompt gives the context that will be used throughout the conversation. It is essneitally the first prompt. The user role is now asking specific questions. "You are an assistant to a microfinance loan officer in Ghana. Be factual, neutral, and never invent details not present in the source text". is an example of a system prompt. 
For the user we have for example: "Summarize this loan application"
A token is text the model processes as a unit. It's essentially billing mechanism. Usually, llms bill based on tokens. longer tokens or longer sentences or prompts will attract higher costs. 

### Part 1.2 — Temperature: the randomness dial

In [4]:
import time
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
test_question = "Suggest a name for a brand new hair salon in Accra."



# TODO: Print all 10 answers, grouped by temperature.
for i in range(1,6):
    response = ask_llm(test_question, temperature=0.0)
    print(f"Temperature 0.0, Attempt {i}: {response.choices[0].message.content}")


for i in range(1,6):
    response = ask_llm(test_question, temperature=1.2)
    print(f"Temperature 1.2, Attempt {i}: {response.choices[0].message.content}")
    

Temperature 0.0, Attempt 1: Here are a few suggestions for a brand new hair salon in Accra:

1. **AfroLocks**: A playful name that celebrates African hair and culture.
2. **GlamourHub Accra**: A name that evokes style, sophistication, and a central location.
3. **Kente Kutz**: "Kente" is a traditional Ghanaian cloth, and "Kutz" is a playful take on "cuts," making this name a fun, cultural reference.
4. **Sankofa Styles**: "Sankofa" is a Ghanaian word that means "go back and fetch it," which could represent a salon that helps clients revisit and revamp their style.
5. **Accra Hair Co.**: A simple, modern name that clearly communicates the salon's location and focus.
6. **Adorn Beauty Studio**: "Adorn" means to decorate or embellish, which is perfect for a salon that helps clients enhance their natural beauty.
7. **Roots Hair Salon**: A name that nods to the idea of hair having roots, while also referencing the cultural heritage of Ghana.

Choose the one that resonates with your vision a

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*
with the temperature 0.0, they had a consistent way the names were. Temperature 1.2 varied by a lot however, there was a lot of randomness. I would go for temperature 0.0 in the loan decision sense because we wouldn't want names that vary too much to affect the decision that would be produced.
> **Answer:** [Double-click to edit]
At temperature 0.0, all five answers were identical. At temperature 1.2, there were more unusual suggestions. 
The reason is that temperature controls how the next token is picked. At 0.0 it will pick the same thing everytime.
For a loan system, it is appropriate to use 0.0 because we should have one correct answer based on the information presented.



---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [5]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [6]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this: {letter}"

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_SYSTEM_V2 = """"You are an assistant to a microfinance loan officer in Ghana.
Summarize loan applications in 3-4 sentences, factually and neutrally.
Only use information stated in the letter — never invent details.
If the amount, income, collateral or repayment plan is not stated, say so.
Do not recommend approval or rejection."""


SUMMARY_PROMPT_V2 = "Summarize this loan application : {letter}"

for letter_id in ["L002", "L006"]:
  letter = LETTERS[letter_id]

  v1 = ask_llm(SUMMARY_PROMPT_V2.format(letter=letter), temperature=0.0) 
  v2 = ask_llm(SUMMARY_PROMPT_V2.format(letter=letter),
                 system_prompt=SUMMARY_SYSTEM_V2,
                 temperature=0.0)


  print("V1 output for letter", letter_id, ":\n", v1.choices[0].message.content)
  print("V2 output for letter", letter_id, ":\n", v2.choices[0].message.content)
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

V1 output for letter L002 :
 Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. He needs the funds to repair his vehicle's engine and pay off personal debts. His business has been slow, but he expects it to improve after the festive season. He doesn't have collateral to offer and is relying on his future earnings to repay the loan. He is seeking urgent assistance.
V2 output for letter L002 :
 Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. His business has been slow, but he expects it to improve after the festive season. The applicant does not have collateral to offer. The repayment plan and his income are not stated in the application.
V1 output for letter L006 :
 Here's a summary of Kofi's loan application:

* Amount requested: GHS 50,000
* Proposed businesses: car washing, provision shop, and importing phones from Dubai
* Applicant's age: 22
* Repayment plan: 1 

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** [Double-click to edit]

V1 tries to invent a repayment plan for Kwame when he actually did not have one. V2 however clearly states that there is no repayment plan.

V1 also dropped details like as in first we had trotro engine being talked about but its changed to vehicle's engine. V2 keeps the wording though. It is necessary for trotro to still come in because it is not just any regular vehicle but an income generating one.


2. The failure mode is called hallucination. No invented details is important because the officer is reading the brief hence putting payment plans when there are no payment plans will cause issues in decision making. The lending agency may end up giving money wrongly based on invalid context given by inventing repayment plans or other details.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [7]:
import json
import pandas as pd

EXTRACT_PROMPT = """Return only  JSON object with these keys: applicant_name(string), amount_ghs (number), purpose (string), monthly_profit_ghs(number or null), has_collateral_or_guarantor(boolean), repayment_months(number or null).

If a field is not stated in the letter, use null. Do not guess.

An example of a letter: I am Kingsley and I do carpentry work in Tamale. I need GHS 10,000 for equipment.

Example of an output:
{{"applicant_name": "Kingsley", "amount_ghs": 10000, "purpose": "equipment", "monthly_profit_ghs": null, "has_collateral_or_guarantor": false, "repayment_months": null}}

Letter: {letter}
  """


def extract_fields(letter_text):
    response = ask_llm(EXTRACT_PROMPT.format(letter=letter_text), temperature=0.0)
    text = response.choices[0].message.content.strip()

    # Strip ```json fences if the model added them.
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("json", "", 1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print(f"Warning: could not parse JSON. Got: {text}")
        return None


# Run on all six letters and collect into a DataFrame.
rows = []
extracted_data = {}          # keep the raw JSON too — Part 3.3 needs it
for letter_id, letter in LETTERS.items():
    fields = extract_fields(letter)
    extracted_data[letter_id] = fields
    # Keep a row even if parsing failed, so all six letters appear.
    rows.append({"letter_id": letter_id, **(fields or {})})

extracted_df = pd.DataFrame(rows)
display(extracted_df)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [Double-click to edit]
1. Doing that would be handing the model the correct answer in the prompt. It's basically like making the model overfit or cheat since information is being leaked. 

2. Removing the instruction actually didn't change anything. The extracted fields were still identical. the null points were done properly. Under normal circumstances though it is advisable to explicitly state this because some models would actually hallucinate. 

3. Temperature controls how much randomness goes into picking a token. If the temperature is 0 that means it'll just pick the highest-probability option and hence the same output everytime which is what we want for extraction. For creativity we would like a blend or variety and not just the most probable one everytime. 

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [8]:
BRIEF_SYSTEM = """You are an assistant to a microfinance loan officer in Ghana. Your function is to produce decision-support briefs. Final decisions are made by the human loan officer, not by you. No decision making, do not output approve or reject or recommend approval outcomes. Base your write up only on the letter and the extracted data, do not invent any details, or give amounts that were not in the writeup. Any information missing should be stated as is."""

BRIEF_PROMPT = """Letter: {letter}

Extracted data: {extracted}

Write a decision-support brief with exactly these four sections: 
1. Strengths(bullet points, grounded in the letter) 
2. Risks (bullet points, grounded in the letter) 
3. Missing information the officer should request 
4. Suggested next step - one of: "invite for interview", "request documents", "flag for senior review". Do not output approve or reject. 

"""


def generate_brief(letter_text, extracted):
  prompt = BRIEF_PROMPT.format(letter=letter_text,
                               extracted=json.dumps(extracted, indent=2))
  response = ask_llm(prompt, system_prompt=BRIEF_SYSTEM, temperature=0.0)
  return response.choices[0].message.content


briefs = {}
for letter_id, letter in LETTERS.items():
  briefs[letter_id] = generate_brief(letter, extracted_data[letter_id])

for letter_id in ["L003", "L006"]:
  print(letter_id)
  print(briefs[letter_id])
  print()

L003
## 1. Strengths
* The applicant, Efua Darko, has a registered business, Darko Fashions, indicating a level of legitimacy and commitment to the venture.
* The business has a history of generating significant revenue, with December revenue alone being GHS 22,000.
* The applicant has a fixed deposit of GHS 5,000 that can be pledged as collateral, reducing the risk of loan default.
* The applicant has provided sales records for the past 18 months, which can be used to assess the business's financial health and stability.

## 2. Risks
* The loan amount of GHS 15,000 is substantial, and the repayment plan of GHS 1,100 monthly for 15 months may pose a financial strain on the business if sales do not meet expectations.
* The business's profitability may be seasonal, with a significant portion of revenue generated during the Christmas season, which could impact the ability to repay the loan during slower months.
* The applicant's reliance on apprentices may pose a risk if they leave or are

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [Double-click to edit]

For L003, the system does well by making sure it listed every strength. There was nothing inferred. and it also correctly identified the missing piece which was Efua's existing debt obligations. 

For L006, the risks were right was well it flags the no experience, no collateral, and repayment resting on businesses. It fails at the strengths because it said stuff like him being young and full of energy without concrete evidence. We see that when a letter contains verifiable facts the system is able to report them accurately but when it doesn't it attempts to manufacture strengths. 

2. We forbid approve/reject because the model does not have every information needed to actually take a decision. That lies in the hands of an authority of the company who might have other information such as the institution's criteria and exceptions and what not. 
There is a possibility of automation bias. These decisions affect livelihoods and because they are automated by technology, officers may end up not cross checking. 


### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [9]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
# Compare the extracted fields to the gold labels for L001, L003 and L006.
FIELDS = ["applicant_name", "amount_ghs", "purpose",
          "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]


def matches(extracted, gold, field):
    # Both missing counts as correct — null was the right answer.
    if pd.isna(extracted) and gold is None:
        return True
    if pd.isna(extracted) or gold is None:
        return False

    if field == "applicant_name":
        return str(extracted).strip().lower() == str(gold).strip().lower()

    # purpose is free text, so exact matching is unfair. Count it correct if
    # most of the meaningful words in the gold answer appear in the extraction.
    if field == "purpose":
        gold_words = [w for w in str(gold).lower().replace("/", " ").replace(",", " ").split()
                      if len(w) > 3]
        found = [w for w in gold_words if w in str(extracted).lower()]
        return len(found) / len(gold_words) >= 0.5

    return extracted == gold


results = {}
for field in FIELDS:
    row = {}
    for letter_id in GOLD:
        extracted = extracted_df.set_index("letter_id").loc[letter_id, field]
        row[letter_id] = matches(extracted, GOLD[letter_id][field], field)
    row["accuracy"] = sum(row.values()) / len(GOLD)
    results[field] = row

accuracy_df = pd.DataFrame(results).T
display(accuracy_df)
print(f"Overall accuracy: {accuracy_df['accuracy'].mean():.0%}")


,L001,L003,L006,accuracy
applicant_name,True,True,True,1.0
amount_ghs,True,True,True,1.0
purpose,True,True,True,1.0
monthly_profit_ghs,True,True,True,1.0
has_collateral_or_guarantor,True,True,True,1.0
repayment_months,True,True,True,1.0


Overall accuracy: 100%


### Part 4.2 — Reliability: is the system consistent?

In [10]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

# Run the extractor on L004 five times at each temperature and count unique results.
for temperature in [0.0, 1.0]:
    results = []
    for i in range(5):
        response = ask_llm(EXTRACT_PROMPT.format(letter=LETTERS["L004"]),
                           temperature=temperature)
        text = response.choices[0].message.content.strip()
        if text.startswith("```"):
            text = text.strip("`").replace("json", "", 1).strip()
        try:
            results.append(json.dumps(json.loads(text), sort_keys=True))
        except json.JSONDecodeError:
            results.append("INVALID JSON")

    valid = [r for r in results if r != "INVALID JSON"]
    print(f"Temperature {temperature}")
    print(f"valid JSON:     {len(valid)}/5")
    print(f"  unique results: {len(set(valid))}")
    for result in set(valid):
        print(f"    {result}")

Temperature 0.0
valid JSON:     5/5
  unique results: 2
    {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "feed and 500 new layers for poultry farm", "repayment_months": 18}
    {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "feed and 500 new layers", "repayment_months": 18}
Temperature 1.0
valid JSON:     5/5
  unique results: 2
    {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "feed and 500 new layers for poultry farm", "repayment_months": 18}
    {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "feed and 500 new layers", "repayment_months": 18}


### Part 4.3 — Hallucination probing

In [11]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

question = ("Summarize this loan application and state the applicant's credit score:\n\n"
            + LETTERS["L001"])
response = ask_llm(question, system_prompt=SUMMARY_SYSTEM_V2, temperature=0.0)
print("TEST 1 — credit score (not in L001)")
print(response.choices[0].message.content)

# Test 2 — feed the extractor irrelevant text.
weather = ("Accra weather today: mostly cloudy with a high of 31 degrees and a "
           "60% chance of afternoon rain. Winds light from the southwest.")
print("\nTEST 2 — weather report through the extractor")
print(extract_fields(weather))

TEST 1 — credit score (not in L001)
Akosua Mensah is applying for a loan of GHS 8,000 to purchase a deep freezer and expand her business. She has a monthly profit of GHS 900 from her current stall and proposes to repay the loan with monthly installments of GHS 450 over 20 months. The applicant has GHS 2,500 in savings with the susu scheme and has a guarantor, her sister, who is a teacher. The applicant's credit score is not stated in the letter.

TEST 2 — weather report through the extractor
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': None}


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

1. Extraction accuracy was 100 percent. For my approach, "buy a deep freezer and expand into frozen foods" was made to be against a gold label of "buy deep freezer/expand into frozen foods". I did not use exact string comparison since this won't be as good as using a check to see if most of the words appear in the extraction.
'purpose' was the hardest field. This is because the other fields have one correct form or answer however for this it can be phrased in different ways.
2. At temperature 0: there was 1 unique result and it returned valid JSONs for all instances.
Both temperatures worked because with temperature 0 we had the same output all the time. With temperature, 1.0 there was no chaos. Json validity held at 5/5 and the variatio nwas just confined to the free-text field.
For production the implication is still to run extraction at temperature 0, because there is no
upside to variation when the answer is fixed and there is real downside in reproducibility. But
five runs on one letter is a small sample. Nothing here proves JSON validity would hold at 5/5
across a thousand letters, which is why the try/except around json.loads stays in regardless
of temperature.

3. Test 1 passed. Asked for Akosua's credit score, which L001 never mentions, the summarizer wrote: "The applicant's credit score is not stated in the letter." It answered naming the basence rather than filling it, and the rest of the summary was accurate. 

Test 2 - partially passed. most of the fields that were null came null however has_collateral_or_guarantor came as false instead of null. This is problematic because these are not the same thing. It is however expected because of the nature of the schema.


In [12]:
r1 = ask_llm(EXTRACT_PROMPT.format(letter=LETTERS["L001"]), temperature=0.0)
print("extraction:", r1.usage)

r2 = ask_llm(BRIEF_PROMPT.format(letter=LETTERS["L001"],
                                 extracted=json.dumps(extracted_data["L001"], indent=2)),
             system_prompt=BRIEF_SYSTEM, temperature=0.0)
print("brief:", r2.usage)

extraction: CompletionUsage(completion_tokens=76, prompt_tokens=324, total_tokens=400, completion_time=0.146024627, completion_tokens_details=None, prompt_time=0.016631857, prompt_tokens_details=None, queue_time=0.113608788, total_time=0.162656484)
brief: CompletionUsage(completion_tokens=321, prompt_tokens=413, total_tokens=734, completion_time=1.057021512, completion_tokens_details=None, prompt_time=0.052732911, prompt_tokens_details=None, queue_time=0.051792913, total_time=1.109754423)


### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit] 

1. The system is reading letters and only knows what it can from the letters and these letters may not have enough about the business. A trader who has been in Makola for over 10 years, with steady income and a solid guarantor will not be granted the loan if their letter is not good enough. The system too as stated earlier just reds the leltters and hence won't be able to distinguish the weakness of a business from the thinness of the letter.

Generally those harmed will be people with less formal education. From the results we see the longer written letter that had hearsay stuff still looked better than a poorly written one.


2. Sending letters to a third party API like Groq means personal information is leaving Ghana to another country's jursidiction. The instutiion therefore will need to have legal backing to do this before sending such information outside. An alternative would be to strip away real names, when sending this information. This will reduce exposure but might cost some context. 

3. Mandatory human review : A human in the loop or a human should be precent an officer who sees the brief generated next to the original letter if required so as to make right decisions and refer to the original letters if required.(the whole process will not be automated or decisions will not just be made based on the brief alone all the time.)
Logging: Logging being important because if everything is logged including the prompt and so on if someone is wrongly declined there is a way to trace and make changes to the prompt. 

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]


1. The loop is the same where we change one thing, re run  measure against some labels and keep or revert. Prompts are dsicrete text but hyperparameters were numbers sitting in a range. with lab3, there were loss curves but here it is outputs I need to inspect.

2. No, I would not trust this system unattended. THis is because in L006's strengths section, it listed a lot of strengths as facts which should not have been the case since they were more "hearsay" without concrete evidence. The problem therefore, is that when the input is thin, the system tries to fill spaces.

3. Cost and scale: one application costs two calls. Groq's rate is about less than a dollar a month for about 1,000 applications. 

4. Training from scratch needs lots of labelled examples which does not exist. Calling an API wins because of this. In Lab2 and Lab3 we needed so many labelled examples to be able to learn anything. This is not needed when you can make an API call to an already trained model. However, there are times were training a model is important this is when feeding data into third party API's is illegal or when it is a very small task. 

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.